# Loan Default Prediction − XGBoost & MLP

In [1]:
from pathlib import Path
import random

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from xgboost import XGBClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = Path("data.csv")
OUTPUT_DIR = Path("output") / "model"
OUTPUT_DIR.mkdir(exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Expected the labelled data file at {DATA_PATH.resolve()}."
    )

## Shared Preprocessing

In [2]:
df = pd.read_csv(DATA_PATH)
required = {"TARGET", "DAYS_EMPLOYED", "DAYS_BIRTH", "AMT_CREDIT", "AMT_INCOME_TOTAL", "AMT_ANNUITY", "AMT_GOODS_PRICE"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

if "CODE_GENDER" in df:
    df = df[df["CODE_GENDER"] != "XNA"]
if "NAME_FAMILY_STATUS" in df:
    df = df[df["NAME_FAMILY_STATUS"] != "Unknown"]
df = df.reset_index(drop=True)

df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
df["age_years"] = -df["DAYS_BIRTH"] / 365.25
for col in ["DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE"]:
    df[col] = -df[col]
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].apply(
    lambda x: -x if pd.notna(x) and x < 0 else x
)

df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["CREDIT_TERM"] = df["AMT_CREDIT"] / df["AMT_ANNUITY"]
df["CREDIT_GOODS_RATIO"] = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"]
ratio_cols = ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM", "CREDIT_GOODS_RATIO"]
df[ratio_cols] = df[ratio_cols].replace([np.inf, -np.inf], np.nan)

COLS_TO_DROP = [
    "SK_ID_CURR", "DAYS_BIRTH", "FLAG_MOBIL", "FLAG_CONT_MOBILE",
    "REGION_RATING_CLIENT_W_CITY", "OWN_CAR_AGE",
    "APARTMENTS_AVG", "APARTMENTS_MEDI", "APARTMENTS_MODE",
    "BASEMENTAREA_AVG", "BASEMENTAREA_MEDI", "BASEMENTAREA_MODE",
    "YEARS_BEGINEXPLUATATION_AVG", "YEARS_BEGINEXPLUATATION_MEDI", "YEARS_BEGINEXPLUATATION_MODE",
    "YEARS_BUILD_AVG", "YEARS_BUILD_MEDI", "YEARS_BUILD_MODE",
    "COMMONAREA_AVG", "COMMONAREA_MEDI", "COMMONAREA_MODE",
    "ELEVATORS_AVG", "ELEVATORS_MEDI", "ELEVATORS_MODE",
    "ENTRANCES_AVG", "ENTRANCES_MEDI", "ENTRANCES_MODE",
    "FLOORSMAX_AVG", "FLOORSMAX_MEDI", "FLOORSMAX_MODE",
    "FLOORSMIN_AVG", "FLOORSMIN_MEDI", "FLOORSMIN_MODE",
    "LANDAREA_AVG", "LANDAREA_MEDI", "LANDAREA_MODE",
    "LIVINGAPARTMENTS_AVG", "LIVINGAPARTMENTS_MEDI", "LIVINGAPARTMENTS_MODE",
    "LIVINGAREA_AVG", "LIVINGAREA_MEDI", "LIVINGAREA_MODE",
    "NONLIVINGAPARTMENTS_AVG", "NONLIVINGAPARTMENTS_MEDI", "NONLIVINGAPARTMENTS_MODE",
    "NONLIVINGAREA_AVG", "NONLIVINGAREA_MEDI", "NONLIVINGAREA_MODE",
    "FONDKAPREMONT_MODE", "HOUSETYPE_MODE", "TOTALAREA_MODE", "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE",
    "FLAG_DOCUMENT_2", "FLAG_DOCUMENT_4", "FLAG_DOCUMENT_7", "FLAG_DOCUMENT_9", "FLAG_DOCUMENT_10",
    "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_13", "FLAG_DOCUMENT_14", "FLAG_DOCUMENT_15",
    "FLAG_DOCUMENT_17", "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21",
]
df = df.drop(columns=[c for c in COLS_TO_DROP if c in df], errors="ignore")

for col, cap in {
    "CNT_CHILDREN": 10,
    "CNT_FAM_MEMBERS": 10,
    "OBS_30_CNT_SOCIAL_CIRCLE": 30,
    "OBS_60_CNT_SOCIAL_CIRCLE": 30,
    "AMT_REQ_CREDIT_BUREAU_QRT": 20,
}.items():
    if col in df:
        df[col] = df[col].clip(upper=cap)

X = df.drop(columns="TARGET")
y = df["TARGET"]
X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
X_train_raw, X_val_raw, X_test_raw = [part.copy() for part in (X_train_raw, X_val_raw, X_test_raw)]

ordinal_cols = [c for c in ["NAME_EDUCATION_TYPE"] if c in X_train_raw]
onehot_cols = [c for c in [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    "NAME_TYPE_SUITE", "NAME_INCOME_TYPE", "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE", "WEEKDAY_APPR_PROCESS_START", "ORGANIZATION_TYPE",
    "OCCUPATION_TYPE",
] if c in X_train_raw]
monetary_cols = [c for c in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"] if c in X_train_raw]
ext_source_cols = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in X_train_raw]

for part in (X_train_raw, X_val_raw, X_test_raw):
    for col in ext_source_cols:
        part[f"{col}_MISSING"] = part[col].isna().astype(int)
    for col in ["OCCUPATION_TYPE", "NAME_TYPE_SUITE"]:
        if col in part:
            part[col] = part[col].fillna("Missing")

RARE_THRESHOLD = 0.01
HIGH_CARD_COLS = ["ORGANIZATION_TYPE", "NAME_INCOME_TYPE", "OCCUPATION_TYPE"]
PROTECTED_CATS = {"NAME_INCOME_TYPE": ["Maternity leave", "Unemployed"]}

for col in HIGH_CARD_COLS:
    if col not in X_train_raw:
        continue
    frequencies = X_train_raw[col].value_counts(normalize=True)
    protected = PROTECTED_CATS.get(col, [])
    rare = frequencies[
        (frequencies < RARE_THRESHOLD) & (~frequencies.index.isin(protected))
    ].index.tolist()
    for part in [X_train_raw, X_val_raw, X_test_raw]:
        part[col] = part[col].apply(lambda value: "Other" if value in rare else value)

ext_missing_flags = [f"{col}_MISSING" for col in ext_source_cols]
passthrough_cols = [c for c in ext_missing_flags + ["DAYS_EMPLOYED_ANOM"] if c in X_train_raw]
all_cat = set(ordinal_cols + onehot_cols)
all_special_numeric = set(monetary_cols + ext_source_cols + passthrough_cols)
numeric_cols = [
    c for c in X_train_raw.columns
    if c not in all_cat
    and c not in all_special_numeric
    and X_train_raw[c].dtype in [np.float64, np.int64, np.float32, np.int32]
]

winsor_caps = {col: X_train_raw[col].quantile(0.99) for col in monetary_cols}
def winsorize_monetary(X, caps=winsor_caps, cols=monetary_cols):
    X = X.copy()
    for i, col in enumerate(cols):
        X[:, i] = np.clip(X[:, i], a_min=None, a_max=caps[col])
    return X

def log1p_transform(X):
    return np.log1p(np.abs(X))

monetary_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("winsorize", FunctionTransformer(winsorize_monetary, validate=False, feature_names_out="one-to-one")),
    ("log1p", FunctionTransformer(log1p_transform, validate=False, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])
standard_numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
ordinal_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OrdinalEncoder(
        categories=[["Lower secondary", "Secondary / secondary special", "Incomplete higher", "Higher education", "Academic degree"]],
        handle_unknown="use_encoded_value", unknown_value=-1,
    )),
])
onehot_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")),
])

preprocessor = ColumnTransformer([
    ("monetary", monetary_pipeline, monetary_cols),
    ("ext_source", standard_numeric_pipeline, ext_source_cols),
    ("ordinal", ordinal_pipeline, ordinal_cols),
    ("onehot", onehot_pipeline, onehot_cols),
    ("numeric", standard_numeric_pipeline, numeric_cols),
    ("passthrough", "passthrough", passthrough_cols),
], remainder="drop", verbose_feature_names_out=False)

X_train_proc = preprocessor.fit_transform(X_train_raw)
X_val_proc = preprocessor.transform(X_val_raw)
X_test_proc = preprocessor.transform(X_test_raw)
feature_names_raw = preprocessor.get_feature_names_out()
feature_names = list(feature_names_raw)

X_train = pd.DataFrame(X_train_proc, columns=feature_names)
X_val = pd.DataFrame(X_val_proc, columns=feature_names)
X_test = pd.DataFrame(X_test_proc, columns=feature_names)

assert not X_train.isna().any().any() and not np.isinf(X_train_proc).any()
print(f"train={X_train.shape}, validation={X_val.shape}, test={X_test.shape}")
print(f"default rates: {y_train.mean():.4f}, {y_val.mean():.4f}, {y_test.mean():.4f}")

train=(215253, 113), validation=(46126, 113), test=(46126, 113)
default rates: 0.0807, 0.0807, 0.0807


In [3]:
def validation_threshold(y_true, scores):
    false_positive_rate, true_positive_rate, candidates = roc_curve(y_true, scores)
    finite = np.isfinite(candidates)
    best = np.flatnonzero(finite)[np.argmax((true_positive_rate - false_positive_rate)[finite])]
    return float(candidates[best])


def evaluate_probabilities(name, y_true, probabilities, threshold):
    predictions = (np.asarray(probabilities).ravel() >= threshold).astype(int)
    metrics = {
        "model": name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
    }
    print(pd.Series(metrics))
    print(classification_report(y_true, predictions, digits=4))
    return metrics

## XGBoost Model

In [4]:
RUN_XGB_TUNING = False
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

xgb_params = {
    "colsample_bytree": 0.7229163764267956,
    "learning_rate": 0.03899272558722083,
    "max_depth": 5,
    "min_child_weight": 8,
    "n_estimators": 472,
    "subsample": 0.8404460046972835,
}

if RUN_XGB_TUNING:
    search_space = {
        "n_estimators": randint(100, 501),
        "max_depth": randint(3, 9),
        "learning_rate": uniform(0.01, 0.29),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.5, 0.5),
        "min_child_weight": randint(1, 11),
    }
    search_model = XGBClassifier(
        scale_pos_weight=scale_pos_weight, eval_metric="auc", tree_method="hist",
        random_state=SEED, n_jobs=-1, verbosity=0,
    )
    xgb_search = RandomizedSearchCV(
        search_model, param_distributions=search_space, n_iter=20,
        scoring="roc_auc", cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        n_jobs=-1, verbose=2, random_state=SEED,
    )
    xgb_search.fit(X_train, y_train)
    xgb_params = xgb_search.best_params_
    print("Best CV ROC-AUC:", xgb_search.best_score_)
    print("Best parameters:", xgb_params)

xgb_model = XGBClassifier(
    **xgb_params, scale_pos_weight=scale_pos_weight, eval_metric="auc",
    tree_method="hist", random_state=SEED, n_jobs=-1, verbosity=0,
)
xgb_model.fit(X_train, y_train)

xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]
xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_threshold = validation_threshold(y_val, xgb_val_prob)
print(f"XGBoost threshold selected on validation: {xgb_threshold:.6f}")
xgb_val_metrics = evaluate_probabilities("XGBoost validation", y_val, xgb_val_prob, xgb_threshold)
xgb_test_metrics = evaluate_probabilities("XGBoost test", y_test, xgb_test_prob, xgb_threshold)
joblib.dump(xgb_model, OUTPUT_DIR / "xgboost_model.pkl")

XGBoost threshold selected on validation: 0.470434
model        XGBoost validation
threshold              0.470434
roc_auc                0.766379
pr_auc                  0.25274
dtype: object
              precision    recall  f1-score   support

           0     0.9639    0.6884    0.8032     42402
           1     0.1660    0.7065    0.2689      3724

    accuracy                         0.6898     46126
   macro avg     0.5650    0.6974    0.5360     46126
weighted avg     0.8995    0.6898    0.7600     46126

model        XGBoost test
threshold        0.470434
roc_auc           0.76893
pr_auc           0.258533
dtype: object
              precision    recall  f1-score   support

           0     0.9641    0.6877    0.8028     42402
           1     0.1662    0.7086    0.2692      3724

    accuracy                         0.6894     46126
   macro avg     0.5651    0.6982    0.5360     46126
weighted avg     0.8997    0.6894    0.7597     46126



['output/model/xgboost_model.pkl']

## MLP Neural Network

In [5]:
X_train_arr = X_train.values.astype(np.float32)
X_val_arr = X_val.values.astype(np.float32)
X_test_arr = X_test.values.astype(np.float32)
y_train_arr = y_train.values.astype(np.float32)
y_val_arr = y_val.values.astype(np.float32)
y_test_arr = y_test.values.astype(np.float32)
N_FEATURES = X_train_arr.shape[1]


def build_mlp_leaky_relu(
    n_features,
    hidden_units=(128, 64),
    dropout_rate=0.3,
    use_batchnorm=True,
    learning_rate=0.001,
    alpha=0.1,
):
    keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = keras.Sequential(name="MLP_LeakyReLU")
    model.add(layers.Input(shape=(n_features,)))

    for units in hidden_units:
        model.add(layers.Dense(units, use_bias=not use_batchnorm))
        if use_batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.LeakyReLU(alpha=alpha))
        model.add(layers.Dropout(dropout_rate, seed=SEED))

    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")],
    )
    return model


BEST_UNITS = (128, 64)
BEST_DROPOUT = 0.5
BEST_LR = 0.001


def train_best_mlp(X_train_data, y_train_data, label="original"):
    model = build_mlp_leaky_relu(
        N_FEATURES,
        hidden_units=BEST_UNITS,
        dropout_rate=BEST_DROPOUT,
        use_batchnorm=True,
        learning_rate=BEST_LR,
        alpha=0.1,
    )
    es = EarlyStopping(
        monitor="val_auc",
        patience=10,
        mode="max",
        restore_best_weights=True,
        verbose=0,
    )
    print(f"Training best MLP on {label} set (n={len(X_train_data):,})")
    history = model.fit(
        X_train_data,
        y_train_data,
        validation_data=(X_val_arr, y_val_arr),
        epochs=100,
        batch_size=1024,
        callbacks=[es],
        verbose=0,
    )
    return model, history


mlp_model, mlp_history = train_best_mlp(X_train_arr, y_train_arr, label="original")
mlp_val_prob = mlp_model.predict(X_val_arr, verbose=0).ravel()
mlp_test_prob = mlp_model.predict(X_test_arr, verbose=0).ravel()
mlp_threshold = validation_threshold(y_val_arr, mlp_val_prob)
print(f"MLP threshold selected on validation: {mlp_threshold:.6f}")
mlp_val_metrics = evaluate_probabilities("MLP validation", y_val, mlp_val_prob, mlp_threshold)
mlp_test_metrics = evaluate_probabilities("MLP test", y_test, mlp_test_prob, mlp_threshold)
mlp_model.save(OUTPUT_DIR / "mlp_model.keras")

Training best MLP on original set (n=215,253)
MLP threshold selected on validation: 0.092110
model        MLP validation
threshold           0.09211
roc_auc            0.757757
pr_auc             0.232143
dtype: object
              precision    recall  f1-score   support

           0     0.9626    0.6878    0.8023     42402
           1     0.1637    0.6955    0.2650      3724

    accuracy                         0.6885     46126
   macro avg     0.5631    0.6917    0.5337     46126
weighted avg     0.8981    0.6885    0.7590     46126

model        MLP test
threshold     0.09211
roc_auc      0.756598
pr_auc       0.240712
dtype: object
              precision    recall  f1-score   support

           0     0.9622    0.6894    0.8033     42402
           1     0.1635    0.6915    0.2645      3724

    accuracy                         0.6896     46126
   macro avg     0.5629    0.6904    0.5339     46126
weighted avg     0.8977    0.6896    0.7598     46126



In [6]:
results = pd.DataFrame([xgb_test_metrics, mlp_test_metrics]).set_index("model")
display(results.sort_values("roc_auc", ascending=False))

joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.pkl")
joblib.dump(list(feature_names), OUTPUT_DIR / "feature_names.pkl")
results.to_csv(OUTPUT_DIR / "test_metrics.csv")
print(f"Saved models and preprocessing artifacts to {OUTPUT_DIR.resolve()}")

,threshold,roc_auc,pr_auc
model,,,
XGBoost test,0.470434,0.768930,0.258533
MLP test,0.092110,0.756598,0.240712


Saved models and preprocessing artifacts to /Users/beta/Desktop/RDS Audit Project/output/model
